# FASE 4 - SIMULACIONES DEL SISTEMA
**Persona 2 - Luis Enrique Beltran Perdomo**  
**Curso:** Programación - UNAD  
**Código:** 213023

In [ ]:
# =============================================
# CELDA 1: IMPORTACIONES Y CONFIGURACIÓN DE LOGS
# =============================================
import logging
from abc import ABC, abstractmethod
from datetime import datetime

logging.basicConfig(
    filename='software_fj_errors.log',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

print('✅ Configuración de logs lista')

In [ ]:
# =============================================
# CELDA 2: EXCEPCIONES PERSONALIZADAS
# =============================================
class SoftwareFJError(Exception):
    """Clase base para excepciones del sistema."""
    pass

class ReservaInvalidaError(SoftwareFJError):
    """Se lanza cuando los parámetros de una reserva son incorrectos."""
    pass

class ServicioNoDisponibleError(SoftwareFJError):
    """Se lanza cuando un servicio no está disponible."""
    pass

class ClienteInvalidoError(SoftwareFJError):
    """Se lanza cuando los datos del cliente son incorrectos."""
    pass

print('✅ Excepciones personalizadas definidas')

In [ ]:
# =============================================
# CELDA 3: CLASE CLIENTE (Encapsulamiento)
# =============================================
class Cliente:
    """Representa un cliente con encapsulamiento de atributos."""

    def __init__(self, nombre, id_cliente, email):
        self.__nombre = None
        self.__id_cliente = None
        self.__email = None
        self.nombre = nombre
        self.id_cliente = id_cliente
        self.email = email

    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        if not valor or not valor.strip():
            raise ClienteInvalidoError('El nombre no puede estar vacío.')
        self.__nombre = valor.strip().title()

    @property
    def id_cliente(self):
        return self.__id_cliente

    @id_cliente.setter
    def id_cliente(self, valor):
        if not str(valor).isdigit() or len(str(valor)) < 4:
            raise ClienteInvalidoError('El ID debe ser numérico y tener al menos 4 dígitos.')
        self.__id_cliente = str(valor)

    @property
    def email(self):
        return self.__email

    @email.setter
    def email(self, valor):
        if '@' not in valor or '.' not in valor:
            raise ClienteInvalidoError('El email no tiene formato válido.')
        self.__email = valor.lower()

    def __str__(self):
        return f'Cliente: {self.__nombre} | ID: {self.__id_cliente} | Email: {self.__email}'

print('✅ Clase Cliente definida')

In [ ]:
# =============================================
# CELDA 4: CLASE ABSTRACTA SERVICIO + SUBCLASES
# (Herencia y Polimorfismo)
# =============================================
class Servicio(ABC):
    """Clase abstracta base para todos los servicios."""

    def __init__(self, nombre_servicio, costo_base, disponible=True):
        self._nombre_servicio = nombre_servicio
        self._costo_base = costo_base
        self._disponible = disponible

    def verificar_disponibilidad(self):
        if not self._disponible:
            raise ServicioNoDisponibleError(
                f"El servicio '{self._nombre_servicio}' no está disponible."
            )

    @abstractmethod
    def calcular_costo(self):
        """Método abstracto sobrescrito en cada subclase (Polimorfismo)."""
        pass

    @abstractmethod
    def describir(self):
        pass


class ReservaSala(Servicio):
    """Reserva de sala — hereda de Servicio."""

    def __init__(self, horas, capacidad=10, disponible=True):
        super().__init__('Reserva de Sala', 50.0, disponible)
        if not isinstance(horas, (int, float)) or horas <= 0 or horas > 24:
            raise ReservaInvalidaError('Las horas deben ser un número entre 1 y 24.')
        self.__horas = horas
        self.__capacidad = capacidad

    def calcular_costo(self):
        return self._costo_base * self.__horas * 1.20  # recargo 20%

    def describir(self):
        return (f'[Sala] {self.__horas}h | Capacidad: {self.__capacidad} | '
                f'Costo: ${self.calcular_costo():,.2f}')


class AlquilerEquipo(Servicio):
    """Alquiler de equipos — hereda de Servicio."""

    def __init__(self, tipo_equipo, cantidad, disponible=True):
        super().__init__('Alquiler de Equipo', 20.0, disponible)
        if not isinstance(cantidad, int) or cantidad <= 0:
            raise ReservaInvalidaError('La cantidad debe ser un entero mayor a 0.')
        self.__tipo_equipo = tipo_equipo
        self.__cantidad = cantidad

    def calcular_costo(self):
        costo = self._costo_base * self.__cantidad
        return costo * 0.90 if self.__cantidad > 3 else costo  # descuento 10%

    def describir(self):
        return (f'[Equipo] {self.__tipo_equipo} x{self.__cantidad} | '
                f'Costo: ${self.calcular_costo():,.2f}')


class AsesoriaEspecializada(Servicio):
    """Asesoría con experto — hereda de Servicio."""

    def __init__(self, area, experto, sesiones, disponible=True):
        super().__init__('Asesoría Especializada', 80.0, disponible)
        if not isinstance(sesiones, int) or sesiones <= 0:
            raise ReservaInvalidaError('Las sesiones deben ser un entero mayor a 0.')
        self.__area = area
        self.__experto = experto
        self.__sesiones = sesiones

    def calcular_costo(self):
        return self._costo_base * self.__sesiones

    def describir(self):
        return (f'[Asesoría] {self.__area} | Experto: {self.__experto} | '
                f'{self.__sesiones} sesión(es) | Costo: ${self.calcular_costo():,.2f}')

print('✅ Clases Servicio, ReservaSala, AlquilerEquipo, AsesoriaEspecializada definidas')

In [ ]:
# =============================================
# CELDA 5: CLASE RESERVA
# =============================================
class Reserva:
    """Integra Cliente y Servicios en una reserva."""

    _contador = 0

    def __init__(self, cliente):
        if not isinstance(cliente, Cliente):
            raise ClienteInvalidoError('Se requiere un objeto Cliente válido.')
        self.__cliente = cliente
        self.__servicios = []
        self.__fecha = datetime.now()
        Reserva._contador += 1
        self.__id = Reserva._contador

    def agregar_servicio(self, servicio):
        if not isinstance(servicio, Servicio):
            raise TypeError('El objeto debe ser una instancia de Servicio.')
        try:
            servicio.verificar_disponibilidad()
        except ServicioNoDisponibleError as e:
            # Encadenamiento de excepciones
            raise ReservaInvalidaError(
                'No se puede agregar el servicio a la reserva.'
            ) from e
        self.__servicios.append(servicio)

    def mostrar_resumen(self):
        total = sum(s.calcular_costo() for s in self.__servicios)
        print(f"{'='*55}")
        print(f'  RESERVA #{self.__id} — {self.__fecha.strftime("%Y-%m-%d %H:%M")}')
        print(f"{'='*55}")
        print(f'  {self.__cliente}')
        for s in self.__servicios:
            print(f'    • {s.describir()}')
        print(f'  TOTAL: ${total:,.2f}')
        print(f"{'='*55}")
        logging.info(f'Reserva #{self.__id} — {self.__cliente.nombre} — Total: ${total:,.2f}')

print('✅ Clase Reserva definida')

In [ ]:
# =============================================
# CELDA 6: FUNCIÓN DE SIMULACIÓN
# =============================================
def ejecutar_simulacion(numero, descripcion, funcion):
    print(f'\n--- SIM {numero}: {descripcion} ---')
    try:
        funcion()
    except ClienteInvalidoError as e:
        logging.error(f'SIM {numero} - Cliente inválido: {e}')
        print(f'❌ Error de cliente: {e}')
    except ReservaInvalidaError as e:
        causa = e.__cause__
        logging.error(f'SIM {numero} - Reserva inválida: {e} | Causa: {causa}')
        print(f'❌ Error en reserva: {e}')
        if causa:
            print(f'   Causa: {causa}')
    except ServicioNoDisponibleError as e:
        logging.warning(f'SIM {numero} - Servicio no disponible: {e}')
        print(f'❌ Servicio no disponible: {e}')
    except TypeError as e:
        logging.error(f'SIM {numero} - Tipo incorrecto: {e}')
        print(f'❌ Error de tipo: {e}')
    except Exception as e:
        logging.critical(f'SIM {numero} - Error crítico: {e}', exc_info=True)
        print(f'❌ Error inesperado: {e}')
    else:
        print('✅ Simulación exitosa')
        logging.info(f'SIM {numero} completada correctamente')
    finally:
        print('   Recursos liberados.')

print('✅ Función de simulación lista')

In [ ]:
# =============================================
# CELDA 7: SIMULACIONES 1 y 2 — Casos válidos
# =============================================

def sim1():
    cliente = Cliente('Luis Enrique', '12345', 'luis@unad.edu.co')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(ReservaSala(horas=3, capacidad=10))
    reserva.mostrar_resumen()

def sim2():
    cliente = Cliente('Maria Lopez', '67890', 'maria@correo.com')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(AlquilerEquipo('Laptop', 5))  # descuento >3
    reserva.mostrar_resumen()

ejecutar_simulacion(1, 'Reserva de sala válida', sim1)
ejecutar_simulacion(2, 'Alquiler con descuento (>3 equipos)', sim2)

In [ ]:
# =============================================
# CELDA 8: SIMULACIONES 3 y 4 — Casos válidos
# =============================================

def sim3():
    cliente = Cliente('Carlos Perez', '11223', 'carlos@gmail.com')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(AsesoriaEspecializada('Python', 'Dr. Silva', 4))
    reserva.mostrar_resumen()

def sim4():
    cliente = Cliente('Ana Gomez', '44556', 'ana@unad.edu.co')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(ReservaSala(horas=2, capacidad=20))
    reserva.agregar_servicio(AlquilerEquipo('Proyector', 2))
    reserva.agregar_servicio(AsesoriaEspecializada('Redes', 'Ing. Torres', 2))
    reserva.mostrar_resumen()

ejecutar_simulacion(3, 'Asesoría especializada válida', sim3)
ejecutar_simulacion(4, 'Múltiples servicios en una reserva', sim4)

In [ ]:
# =============================================
# CELDA 9: SIMULACIONES 5 y 6 — Errores de cliente
# =============================================

def sim5():
    # Nombre vacío — debe lanzar ClienteInvalidoError
    cliente = Cliente('', '99887', 'pedro@correo.com')
    reserva = Reserva(cliente)
    reserva.mostrar_resumen()

def sim6():
    # Email inválido — debe lanzar ClienteInvalidoError
    cliente = Cliente('Sofia Reyes', '66554', 'correo-sin-arroba')
    reserva = Reserva(cliente)
    reserva.mostrar_resumen()

ejecutar_simulacion(5, 'Error: nombre de cliente vacío', sim5)
ejecutar_simulacion(6, 'Error: email inválido', sim6)

In [ ]:
# =============================================
# CELDA 10: SIMULACIONES 7 y 8 — Errores de servicio
# =============================================

def sim7():
    # Horas negativas — debe lanzar ReservaInvalidaError
    cliente = Cliente('Jorge Mora', '55443', 'jorge@unad.edu.co')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(ReservaSala(horas=-2))
    reserva.mostrar_resumen()

def sim8():
    # Servicio no disponible — encadenamiento de excepciones
    cliente = Cliente('Elena Vargas', '77665', 'elena@unad.edu.co')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(ReservaSala(horas=2, disponible=False))
    reserva.mostrar_resumen()

ejecutar_simulacion(7, 'Error: horas negativas en sala', sim7)
ejecutar_simulacion(8, 'Error: servicio no disponible (encadenamiento)', sim8)

In [ ]:
# =============================================
# CELDA 11: SIMULACIONES 9 y 10 — Más errores
# =============================================

def sim9():
    # Cantidad de equipos cero — debe lanzar ReservaInvalidaError
    cliente = Cliente('Laura Diaz', '33221', 'laura@correo.com')
    reserva = Reserva(cliente)
    reserva.agregar_servicio(AlquilerEquipo('Tablet', 0))
    reserva.mostrar_resumen()

def sim10():
    # Objeto que no es Servicio — debe lanzar TypeError
    cliente = Cliente('Pedro Castro', '88776', 'pedro@unad.edu.co')
    reserva = Reserva(cliente)
    reserva.agregar_servicio('esto no es un servicio')
    reserva.mostrar_resumen()

ejecutar_simulacion(9, 'Error: cantidad de equipos inválida', sim9)
ejecutar_simulacion(10, 'Error: objeto inválido como servicio', sim10)

In [ ]:
# =============================================
# CELDA 12: SIMULACIONES 11 y 12
# =============================================

def sim11():
    # ID de cliente muy corto
    cliente = Cliente('Rosa Jimenez', '12', 'rosa@correo.com')
    reserva = Reserva(cliente)
    reserva.mostrar_resumen()

def sim12():
    # Encadenamiento explícito de excepciones
    try:
        raise ValueError('Valor de entrada corrupto')
    except ValueError as e:
        raise SoftwareFJError('Fallo crítico del sistema') from e

ejecutar_simulacion(11, 'Error: ID de cliente inválido', sim11)
ejecutar_simulacion(12, 'Error encadenado explícito', sim12)

In [ ]:
# =============================================
# CELDA 13: MOSTRAR LOGS
# =============================================
print('\n===== CONTENIDO DEL ARCHIVO DE LOGS =====')
try:
    with open('software_fj_errors.log', 'r') as f:
        contenido = f.read()
        print(contenido if contenido else 'El archivo de logs está vacío')
except FileNotFoundError:
    print('No se encontró el archivo de logs')
print('=========================================')
print('\nSimulaciones completadas por: Luis Enrique Beltran Perdomo')
print('Curso: Programación - UNAD 213023')